In [0]:
%sql
-- ============================================================
-- Inventory Intelligence Workflow — Gold Dev Schema Bootstrap
-- ============================================================
-- Data Engineering owns the core star schema in gold_dev.dim +
-- gold_dev.supply_chain_analytics.
--
-- Following Kimball naming conventions (dim_ / fact_ prefixes):
--   1. quote_metadata         — Teams & Review App metadata per quote header
--   2. dim_bom                — Parent-to-child component breakdown (BOM hierarchy)
--   3. dim_supplier_contract  — Supplier lead times, MOQ, pack sizes, contract terms
--   4. fact_plant_capacity    — Plant production capacity limits per period
-- ============================================================

CREATE SCHEMA IF NOT EXISTS gold_dev.supply_chain_analytics;


In [0]:
%sql
-- ============================================================
-- Table 1: quote_metadata
-- Companion table to fact_restock_request for quote header fields.
-- ============================================================

CREATE OR REPLACE TABLE gold_dev.supply_chain_analytics.quote_metadata (
  quote_id STRING NOT NULL COMMENT 'Business key matching gold_dev.supply_chain_analytics.fact_restock_request.QUOTE_ID',
  summary_report STRING COMMENT 'Genie Agent natural-language assumption/reasoning report for this quote',
  teams_message_id STRING COMMENT 'Reference ID of the Adaptive Card sent to Teams',
  teams_sent_at TIMESTAMP COMMENT 'When the Teams notification was dispatched',
  databricks_preview_url STRING COMMENT 'Deep link to the Databricks Review App for this quote',
  decision_comments STRING COMMENT 'Optional approver comments explaining the decision',
  created_by STRING COMMENT 'Agent that created this quote record (default: supervisor_agent)',
  created_at TIMESTAMP COMMENT 'Row creation timestamp',
  updated_at TIMESTAMP COMMENT 'Last modification timestamp',
  CONSTRAINT pk_quote_metadata PRIMARY KEY (quote_id)
)
COMMENT 'Teams/Review-App metadata per quote header, keyed by quote_id.';

INSERT INTO gold_dev.supply_chain_analytics.quote_metadata VALUES
  (
    'QT-2026-0001',
    'PENDING_APPROVAL: two part-lines flagged, one CRITICAL and one HIGH urgency. Awaiting PM review in the Databricks Review App.',
    'msg-teams-0001',
    '2026-08-17 09:00:00',
    'https://workspace.databricks.com/apps/restock-review?quote_id=QT-2026-0001',
    NULL,
    'supervisor_agent',
    '2026-08-17 08:58:00',
    '2026-08-17 09:00:00'
  ),
  (
    'QT-2026-0006',
    'REJECTED: PM decided the CRITICAL and HIGH urgency lines did not warrant restocking this cycle.',
    'msg-teams-0006',
    '2026-08-17 07:30:00',
    'https://workspace.databricks.com/apps/restock-review?quote_id=QT-2026-0006',
    'Rejected -- alternate supplier already covering this shortfall outside the system.',
    'supervisor_agent',
    '2026-08-17 07:25:00',
    '2026-08-17 09:15:00'
  ),
  (
    'QT-2026-0010',
    'COMPLETED: both part-lines approved and fulfilled; Restock Agent confirmed real-time stock and closed out the quote.',
    'msg-teams-0010',
    '2026-08-17 06:45:00',
    'https://workspace.databricks.com/apps/restock-review?quote_id=QT-2026-0010',
    'Approved -- critical for production line continuity.',
    'supervisor_agent',
    '2026-08-17 06:40:00',
    '2026-08-17 10:20:00'
  );


In [0]:
%sql
-- ============================================================
-- Table 2: dim_bom (Bill of Materials Dimension)
-- Parent-to-child component breakdown for BOM explosion.
-- ============================================================

CREATE OR REPLACE TABLE gold_dev.supply_chain_analytics.dim_bom (
  fg_part_id STRING NOT NULL COMMENT 'Finished good or assembly part ID (matches dim_part.PART_ID)',
  component_part_id STRING NOT NULL COMMENT 'Child component part ID required for assembly (matches dim_part.PART_ID)',
  qty_per_unit INT NOT NULL COMMENT 'Quantity of component required per single unit of finished good',
  created_at TIMESTAMP COMMENT 'Row creation timestamp',
  CONSTRAINT pk_dim_bom PRIMARY KEY (fg_part_id, component_part_id)
)
COMMENT 'Bill of Materials (BOM) parent-child component relationship dimension for manufacturing assembly explosion.';

INSERT INTO gold_dev.supply_chain_analytics.dim_bom VALUES
  ('PART-001', 'PART-002', 4, current_timestamp()),
  ('PART-001', 'PART-003', 2, current_timestamp()),
  ('PART-001', 'PART-004', 12, current_timestamp()),
  ('PART-005', 'PART-006', 2, current_timestamp()),
  ('PART-005', 'PART-007', 8, current_timestamp());


In [0]:
%sql
-- ============================================================
-- Table 3: dim_supplier_contract (Supplier Contract Dimension)
-- Supplier commercial terms, lead times, MOQ, and pack sizes.
-- ============================================================

CREATE OR REPLACE TABLE gold_dev.supply_chain_analytics.dim_supplier_contract (
  part_id STRING NOT NULL COMMENT 'Part business key matching dim_part.PART_ID',
  supplier_id STRING NOT NULL COMMENT 'Supplier business key matching dim_supplier.SUPPLIER_ID',
  lead_time_days INT NOT NULL COMMENT 'Contracted lead time in days for this part-supplier pair',
  moq INT NOT NULL COMMENT 'Minimum Order Quantity (MOQ) required by supplier',
  pack_size INT NOT NULL COMMENT 'Pack size increment (order quantity must be multiple of pack_size)',
  unit_cost DOUBLE COMMENT 'Contracted unit price in INR',
  is_preferred BOOLEAN COMMENT 'True if this is the primary contract supplier',
  created_at TIMESTAMP COMMENT 'Row creation timestamp',
  CONSTRAINT pk_dim_supplier_contract PRIMARY KEY (part_id, supplier_id)
)
COMMENT 'Supplier commercial terms, lead times, MOQ, and pack size constraints per part-supplier pair.';

INSERT INTO gold_dev.supply_chain_analytics.dim_supplier_contract VALUES
  ('PART-001', 'SUPP-001', 7, 500, 100, 450.00, true, current_timestamp()),
  ('PART-001', 'SUPP-002', 15, 1000, 250, 420.00, false, current_timestamp()),
  ('PART-002', 'SUPP-001', 5, 2000, 500, 85.00, true, current_timestamp()),
  ('PART-002', 'SUPP-003', 12, 1000, 200, 90.00, false, current_timestamp()),
  ('PART-003', 'SUPP-002', 10, 300, 50, 1200.00, true, current_timestamp());


In [0]:
%sql
-- ============================================================
-- Table 4: fact_plant_capacity (Plant Capacity Fact Table)
-- Rated production capacity per manufacturing plant & period.
-- ============================================================

CREATE OR REPLACE TABLE gold_dev.supply_chain_analytics.fact_plant_capacity (
  plant_id STRING NOT NULL COMMENT 'Plant business key matching dim_plant.PLANT_ID',
  period_start DATE NOT NULL COMMENT 'Start date of capacity planning period',
  period_end DATE NOT NULL COMMENT 'End date of capacity planning period',
  available_capacity_units INT NOT NULL COMMENT 'Total production capacity in units for this period',
  created_at TIMESTAMP COMMENT 'Row creation timestamp',
  CONSTRAINT pk_fact_plant_capacity PRIMARY KEY (plant_id, period_start)
)
COMMENT 'Plant-level rated manufacturing capacity per time period.';

INSERT INTO gold_dev.supply_chain_analytics.fact_plant_capacity VALUES
  ('PLANT-001', DATE '2026-08-01', DATE '2026-08-31', 7500, current_timestamp()),
  ('PLANT-001', DATE '2026-09-01', DATE '2026-09-30', 8000, current_timestamp()),
  ('PLANT-002', DATE '2026-08-01', DATE '2026-08-31', 12000, current_timestamp()),
  ('PLANT-002', DATE '2026-09-01', DATE '2026-09-30', 12500, current_timestamp());


In [0]:
%sql
-- ============================================================
-- Verification: Summary counts of all 4 tables in gold_dev.supply_chain_analytics
-- ============================================================

SELECT 'quote_metadata' AS table_name, COUNT(*) AS row_count FROM gold_dev.supply_chain_analytics.quote_metadata
UNION ALL
SELECT 'dim_bom' AS table_name, COUNT(*) AS row_count FROM gold_dev.supply_chain_analytics.dim_bom
UNION ALL
SELECT 'dim_supplier_contract' AS table_name, COUNT(*) AS row_count FROM gold_dev.supply_chain_analytics.dim_supplier_contract
UNION ALL
SELECT 'fact_plant_capacity' AS table_name, COUNT(*) AS row_count FROM gold_dev.supply_chain_analytics.fact_plant_capacity;
